# 3. Dimensionality reduction: coarse-graining (MODELLER comparison) <a id="3"></a>
In this section we will apply dimensionality reduction to our coarse-grained representation of selected activation loops.


## Table of contents

- [3.2 Coarse-graining activation loops](#26)
  - [3.2.1 Cα interpolation / coarse-graining](#252fit)
  - [3.2.2 MODELLER vs no-MODELLER fit comparison](#253modeller)


## Backend map

How this notebook connects to `workflow/` modules (arrows point into the notebook; includes transitive `workflow` subdependencies):

![Backend map](images/backend_maps/05b-CoarseGrainingModeller.v2.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
%%{init: {"flowchart": {"nodeSpacing": 12, "rankSpacing": 28, "padding": 4}, "themeVariables": {"fontSize": "11px"}} }%%
flowchart LR
  NB["05b-CoarseGrainingModeller.ipynb"]
  m_ca_stripper["ca_stripper"]
  m_chain_basenames["chain_basenames"]
  m_fitting_class["fitting_class"]
  m_reconstruct["reconstruct"]
  m_utilities["utilities"]
  m_ca_stripper --> m_fitting_class
  m_chain_basenames --> m_utilities
  m_utilities --> m_ca_stripper
  m_fitting_class --> NB
  m_reconstruct --> NB
  m_utilities --> NB
```
-->


![State of the workflow](images/DimensionalityReduction.png)

To get started, let's load some packages!

In [ ]:
import os
from glob import glob
import shutil
from IPython.display import display, HTML

from workflow.fitting_class import Fitting
from workflow.reconstruct import ProteinReconstructor
from workflow.utilities import PDBDownloader
from workflow.utilities import clear_and_make
from workflow.utilities import copy_cg_chain_small_molecules
from workflow.utilities import (
    count_pdb_files,
    braf_res,
    clear_and_make,
    make_seg,
    copy_filtered_pdbs,
    copy_cg_chain_small_molecules,
)


## 3.2 Coarse-graining activation loops <a id="26"></a>
Here we coarse-grain activation loops to a uniform number of backbone points, independently of the original loop length. Full chains in `Results/activation_segments/misaligned_filter/` are CA-stripped on the fly (DFG→APE) and resampled to 27 points.


### 3.2.1 Cα interpolation / coarse-graining  <a id="252fit"></a>


We prepare a uniform input for dimensionality reduction. Loops have different numbers of residues, so each DFG→APE segment is either copied (if it already has 27 Cαs) or cubic-spline fitted and resampled to 27 equidistant points.


We use `Fitting(n_ca_template=27, copy_exact_length=True)` with `process_chains_directory`: it strips the activation loop from each full chain, copies exact-length 27-CA loops unchanged onto the template, and spline-fits all others. Output is written to `Results/activation_segments/fitted/`.


In [ ]:
from workflow.fitting_class import Fitting

CHAINS_DIR = "Results/activation_segments/misaligned_filter/"
FITTED_DIR = "Results/activation_segments/fitted"
N_CG = 27  # number of equidistant backbone points for coarse-graining

# CA-strip the activation loop on-the-fly; 27-CA loops are copied, others are spline-fitted to N_CG points
fitter = Fitting(n_ca_template=N_CG, copy_exact_length=True)
fitter.process_chains_directory(
    input_dir=CHAINS_DIR,
    output_dir=FITTED_DIR,
    motifs=["DFG", "APE"],
)


We also save a copy of the protein–small-molecule complexes for chains in the coarse-grained (fitted) set. Complexes are taken from `Results/motif_filtered_small_molecules/` and written to `Results/CG_chain_small_molecules/`, matched by `PDBID_CHAIN`.


In [ ]:
from workflow.utilities import copy_cg_chain_small_molecules

S_MOLECULES_SRC = "Results/motif_filtered_small_molecules/"
S_MOLECULES_DST = "Results/CG_chain_small_molecules/"
FITTED_DIR = "Results/activation_segments/fitted"

copy_cg_chain_small_molecules(
    small_molecules_src=S_MOLECULES_SRC,
    small_molecules_dst=S_MOLECULES_DST,
    cg_dir=FITTED_DIR,
)


### 3.2.2 MODELLER vs no-MODELLER fit comparison  <a id="253modeller"></a>

For loops that are **not** already 27 Cα after DFG→APE stripping, compare cubic-spline fitting **with** vs **without** a prior MODELLER reconstruction of the same chains from `Results/activation_segments/misaligned_filter/`.

Pipeline (cohort only):

1. CA-strip and keep structures with CA count ≠ 27
2. **Arm A** — fit directly (no MODELLER)
3. **Arm B** — MODELLER reconstruct → CA-strip → fit
4. For each basename in both arms, build the 27×27 CA distance matrix and compute
   \(\mathrm{MSE}(D_{\mathrm{noMod}} - D_{\mathrm{Mod}})\) over the full matrix
5. Plot MSE distribution and example heatmaps

Artifacts → `Results/Experiments/modeller_fit_compare/` (production `fitted/` is unchanged).


In [ ]:
from workflow.fitting_class import Fitting
from workflow.reconstruct import ProteinReconstructor
from workflow.utilities import clear_and_make
import shutil

CHAINS_DIR = "Results/activation_segments/misaligned_filter/"
FULL_PDB_DIR = "Results/InterPro_PDBs/"
N_CG = 27

EXP_DIR = "Results/Experiments/modeller_fit_compare"
COHORT_DIR = os.path.join(EXP_DIR, "cohort_chains")
FIT_NO_MOD = os.path.join(EXP_DIR, "fitted_without_modeller")
MOD_OUT = os.path.join(EXP_DIR, "modeller_reconstructed")
FIT_WITH_MOD = os.path.join(EXP_DIR, "fitted_with_modeller")
PLOT_DIR = os.path.join(EXP_DIR, "plots")

for d in (EXP_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)

print("CHAINS_DIR :", CHAINS_DIR)
print("EXP_DIR    :", EXP_DIR)
print("N_CG       :", N_CG)


#### Build non-27-CA cohort

CA-strip every chain in `misaligned_filter/`, keep basenames with loop CA count ≠ 27, and copy those full chains into `cohort_chains/`.


In [ ]:
fitter = Fitting(n_ca_template=N_CG, copy_exact_length=True)
cohort_info = Fitting.list_nonexact_loop_basenames(
    CHAINS_DIR,
    n_ca=N_CG,
    motifs=["DFG", "APE"],
    quiet=True,
)
cohort = cohort_info["nonexact"]
print(f"Cohort size (≠ {N_CG} CA): {len(cohort)}")

clear_and_make(COHORT_DIR)
n_copied = 0
missing = []
for stem in cohort:
    src = os.path.join(CHAINS_DIR, f"{stem}.pdb")
    if not os.path.isfile(src):
        missing.append(stem)
        continue
    shutil.copy2(src, os.path.join(COHORT_DIR, f"{stem}.pdb"))
    n_copied += 1

with open(os.path.join(EXP_DIR, "cohort_basenames.txt"), "w") as fh:
    fh.writelines(s + "\n" for s in cohort)

print(f"Copied {n_copied} cohort chains → {COHORT_DIR}")
if missing:
    print(f"Missing source PDBs: {len(missing)}")
if len(cohort) == 0:
    raise RuntimeError("Empty cohort — nothing to compare")


#### Arm A — fit without MODELLER


In [ ]:
fitter_a = Fitting(n_ca_template=N_CG, copy_exact_length=True)
fitter_a.process_chains_directory(
    input_dir=COHORT_DIR,
    output_dir=FIT_NO_MOD,
    motifs=["DFG", "APE"],
    basename_allowlist=cohort,
    quiet_strip=True,
)
print(f"Arm A fitted PDBs: {len(glob(os.path.join(FIT_NO_MOD, '*.pdb')))}")


#### Arm B — MODELLER reconstruct, then fit

Run MODELLER on the cohort (full PDBs from `Results/InterPro_PDBs/`), normalize output
names to `PDB_CHAIN.pdb`, then apply the same CA-strip + cubic-spline fit.


In [ ]:
clear_and_make(MOD_OUT)
recon = ProteinReconstructor(
    input_dir=COHORT_DIR,
    full_pdb_dir=FULL_PDB_DIR,
    output_dir=MOD_OUT,
    max_gap_length=4,
    max_missing_residues=7,
)
recon.run_modeller_pipeline()

# Normalize MODELLER outputs: *_reconstructed.pdb → PDB_CHAIN.pdb
renamed = 0
for path in glob(os.path.join(MOD_OUT, "*.pdb")):
    base = os.path.basename(path)
    if base.endswith("_reconstructed.pdb"):
        stem = base.replace("_reconstructed.pdb", "")
        dst = os.path.join(MOD_OUT, f"{stem}.pdb")
        if os.path.abspath(path) != os.path.abspath(dst):
            if os.path.isfile(dst):
                os.remove(dst)
            os.rename(path, dst)
            renamed += 1
print(f"Normalized {renamed} *_reconstructed.pdb names")
print(f"MODELLER output PDBs: {len(glob(os.path.join(MOD_OUT, '*.pdb')))}")


In [ ]:
fitter_b = Fitting(n_ca_template=N_CG, copy_exact_length=True)
fitter_b.process_chains_directory(
    input_dir=MOD_OUT,
    output_dir=FIT_WITH_MOD,
    motifs=["DFG", "APE"],
    quiet_strip=True,
)
print(f"Arm B fitted PDBs: {len(glob(os.path.join(FIT_WITH_MOD, '*.pdb')))}")


#### Distance-matrix MSE

For each basename present in both fitted directories, compute the full pairwise CA
distance matrices and \(\mathrm{MSE}(D_{\mathrm{noMod}} - D_{\mathrm{Mod}})\).


In [ ]:
mse_df = Fitting.compare_fitted_distance_matrices(
    FIT_NO_MOD,
    FIT_WITH_MOD,
    label_a="no_modeller",
    label_b="with_modeller",
)
csv_path = os.path.join(EXP_DIR, "distance_matrix_mse.csv")
mse_df.to_csv(csv_path, index=False)
print(f"Saved → {csv_path}")

try:
    from IPython.display import display
    display(mse_df[["basename", "mse", "frobenius", "n_ca"]].describe())
    display(mse_df.sort_values("mse", ascending=False).head(10))
except Exception:
    print(mse_df.head())

if len(mse_df) == 0:
    raise RuntimeError(
        "No overlapping fitted structures — check MODELLER failures / naming."
    )


#### Plots

Histogram and box/strip of per-structure MSE, plus heatmaps of
\(D_{\mathrm{noMod}}\), \(D_{\mathrm{Mod}}\), and \(|D_{\mathrm{noMod}}-D_{\mathrm{Mod}}|\)
for low / median / high MSE examples.


In [ ]:
plot_paths = Fitting.plot_modeller_fit_mse(
    mse_df,
    PLOT_DIR,
    path_col_a="path_no_modeller",
    path_col_b="path_with_modeller",
    n_examples=3,
    show=True,
)
for k, p in plot_paths.items():
    print(f"{k:16s} → {p}")
